# Figure 3 — Primary associations between weather conditions and crash frequency

- **FIG_ID**: `Figure 3`（编号真源：`data/figure_numbering.csv`，§4.0；output stem `Figure_3`）
- **图位**: v6 正文行 208（§4.2）；对应 `[FIGURE 3 PLACEHOLDER]`
- **数据依赖**: `data/fig03_forest_rows.csv`（A 层冻结，5 行；SHA-256 见 `data/figure_data_manifest.json`）
- **上游变更提示**: 若源表更新，先运行 `python scripts/extract_fig03_data.py` 重新冻结，再执行本册
  （Cell 3 会拦截哈希不匹配；`rain_event` 的 CI 溯源见 `fig03_record.md` 与 manifest 的 `ci_provenance`）
- **输出**: `out/draft/Figure_3.{png,pdf,eps}`（`FIG_DRAFT=1`，默认）/ `out/submission/Figure_3.*`
  （`FIG_DRAFT=0`）+ `out/captions/Figure_3_caption.txt` + `qc/figure_3_qc.json`
- **图面约束**（v6 行 208）: 参考线 1.0；每行标注暴露单位、参考组、规格与区间方法；
  强风与冰冻标为未过多重性控制的探索性线索；**图内不出现信息测度、交互估计与预测重要性**
- **设计系统（§1.6，v2.3.1）**: 三列固定布局（森林图 / 变量与暴露定义 / 数值与模型，**数值列右对齐**）；
  主色 NJU purple `#6E0F6D`（主要 + 预先指定的次级），探索性灰 `#7A7A7A`，参考线 `#4A4A4A`，
  网格 `#D9D9D9`；字号取自 `fig_common.DS`（轴 9 / 刻度 8 / 注释 7.5 / 数值 8）；
  **必须带图例说明各层含义**；图内禁用内部代号（`NB2`/`SPEC_P`/`M3`），规格简写为 `Negative binomial; CR1`
- **统计量报告纪律（§1.6.4）**: 同一多重性族内 q 要么全报要么不报；本图四个二分天气主效应为同一族（n = 4），
  统一报 `q`；连续降水行属另一管线，标注为在其自身族内通过控制而不报 q 值


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path


def _resolve_scripts_dir() -> Path:
    """定位 figure/scripts，使本册在**任何内核 cwd** 下都可运行。

    兼容场景：cwd = figure/notebooks（nbconvert 默认）、figure/、仓库根（VS Code / 任意
    启动位置的默认行为）、以及更上层目录；最后回退到本仓库的绝对路径，保证一定能找到
    fig_common.py（旧版写死 `Path.cwd().parent / "scripts"`，只在 notebooks/ 下可用）。
    """
    here = Path.cwd()
    cands = [here / "scripts", here.parent / "scripts"]
    cands += [p / "figure" / "scripts" for p in (here, *here.parents)]
    cands.append(Path(r"D:\NJU_folder\final_update\15_paper\figure\scripts"))
    for c in cands:
        if (c / "fig_common.py").exists():
            return c.resolve()
    raise RuntimeError("未找到 figure/scripts/fig_common.py：请从本仓库、figure/ 或 "
                       "figure/notebooks/ 任一目录启动内核，或检查仓库路径")


SCRIPTS_DIR = _resolve_scripts_dir()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))
print("scripts dir:", SCRIPTS_DIR)

from fig_common import *          # 样式、色板、导出、核验（唯一来源，勿在本册硬编码样式）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f">>> Figure 3 notebook | design system {DESIGN_VERSION} | notebook API v{NOTEBOOK_API} "
      f"| 三列固定布局 + NJU purple")
print(">>> 若本行未出现、或版本号低于上面的值，说明内核里跑的是**旧版 notebook**："
      "请在编辑器执行 Revert File（VS Code）/ 关闭标签后重新打开本文件")

# ---- 本图配置 -------------------------------------------------------------
FIG_ID = "Figure 3"
IS_DRAFT = os.environ.get("FIG_DRAFT", "1") == "1"   # 1 -> out/draft；0 -> out/submission
DATA_FILE = "fig03_forest_rows.csv"
t0 = time.time()
meta = figure_meta(FIG_ID)
print(f"FIG_ID={FIG_ID}  stem={meta['filename_stem']}  IS_DRAFT={IS_DRAFT}")
print("data dir   :", FIG_DATA, "|", DATA_FILE, "存在:", (FIG_DATA / DATA_FILE).exists())
print("输出目录:", FIG_OUT_DRAFT if IS_DRAFT else FIG_OUT_SUB)


In [ ]:
# ---- 数据加载与核验（不匹配即中断）---------------------------------------
report = verify_figure_data(FIG_ID, data_files=[DATA_FILE])
df = pd.read_csv(FIG_DATA / DATA_FILE, encoding="utf-8-sig")

assert len(df) == 5, f"预期 5 行，实际 {len(df)}"
assert set(df.row_key) == {"precip_cont", "rain_event", "low_visibility", "strong_wind", "freezing"}
# 版本守卫：证据层级取值必须与当前绘图代码的 STYLE 键一致。
# 若这里报错且提示出现 'suggestive'/'no_evidence'，或上面的断言出现 'secondary' 的 KeyError，
# 几乎总是"编辑器仍持有旧版 notebook（v2.2 内联两列绘图代码）"——不是数据问题。
EXPECTED_EVIDENCE = {"primary", "secondary", "exploratory"}
_legacy = {"suggestive", "no_evidence"} & set(df.evidence)
assert not _legacy, (
    f"数据文件的 evidence 出现旧版取值 {sorted(_legacy)}，说明 figure\data\fig03_forest_rows.csv 是旧版；"
    "请先运行 python figure\scripts\extract_fig03_data.py 重新冻结")
_unexpected = set(df.evidence) - EXPECTED_EVIDENCE
assert not _unexpected, (
    f"evidence 取值出现未预期的 {sorted(_unexpected)}（当前代码期望 {sorted(EXPECTED_EVIDENCE)}）。"
    "最常见原因：你正在运行**旧版 notebook**——v2.2 初版的 Cell 内联了 `(col, mk) = STYLE[r.evidence]` 两列绘图代码，"
    "其 STYLE 键为 primary/suggestive/exploratory/no_evidence，遇到新数据的 'secondary' 会抛 KeyError。"
    "请从磁盘重新加载 figure\notebooks\fig03_forest.ipynb"
    "（VS Code：命令面板 → File: Revert File，或关闭该标签后重新打开；Jupyter：关闭并重新打开该 notebook）")
# 登记值自检（§2-Fig3 的行与数值表）
REG = {"precip_cont": (1.098, 1.101, 1.042, 1.156),
       "rain_event": (1.465, 1.465, 1.034, 2.075),
       "low_visibility": (1.200, 1.200, 1.086, 1.326),
       "strong_wind": (1.119, 1.119, 1.001, 1.252),
       "freezing": (1.186, 1.186, 0.900, 1.563)}
for _, r in df.iterrows():
    e = REG[r.row_key]
    assert max(abs(r.irr_low - e[0]), abs(r.irr_high - e[1]),
               abs(r.ci_low - e[2]), abs(r.ci_high - e[3])) < 5e-3, f"{r.row_key} 与登记值不符"
# §1.6.4：同一多重性族（四个二分主效应）必须统一报 q
family = df[df.row_key != "precip_cont"]
assert family.q_value.notna().all(), "多重性族内存在未报 q 的行"
assert df[df.row_key == "precip_cont"].q_value.isna().all(), "跨族行不应报 q 值"
# 图内禁用内部代号
for bad in ("NB2", "SPEC_P", "M3"):
    assert not df[["label", "unit", "spec", "stat_note"]].astype(str).apply(
        lambda c: c.str.contains(bad, regex=False)).any().any(), f"图内出现内部代号 {bad}"
print("核验通过：", report["checked"])
df[["order", "row_key", "label", "unit", "reference", "irr_low", "ci_low", "ci_high", "evidence", "stat_note"]]


In [ ]:
# ---- 绘图：三列固定布局森林图（§1.6.8 + §1.6.2 v2.3.1）--------------------
# 面板装配代码放在同图模块 fig03_forest_core.build_figure() 内，
# 使命令行试绘（--draft/--submission）与本册出图共用唯一实现，避免两处拷贝漂移。
from fig03_forest_core import build_figure

fig = build_figure(df)
for ax in fig.get_axes():
    bb = ax.get_position()
    print(f"col x0={bb.x0:.3f} x1={bb.x1:.3f}  width={bb.width:.3f} (figure fraction)")
leg = fig.legends[0]
print("legend entries:", [t.get_text() for t in leg.get_texts()])
fig


In [ ]:
# ---- 导出双版本 + caption + qc -------------------------------------------
written = export_figure(fig, FIG_ID, is_draft=IS_DRAFT)
print("\n".join(str(p) for p in written))

from fig03_forest_core import CAPTION

cap = write_caption(FIG_ID, CAPTION,
                    title="Figure 3. Primary associations between weather conditions and crash frequency.")
print(cap)

qc = fig_qc_dump(FIG_ID, {
    "status": "ok", "is_draft": IS_DRAFT, "seconds": round(time.time() - t0, 1),
    "data_checked": report["checked"], "data_file": DATA_FILE,
    "palette": "NJU purple #6E0F6D (primary + prespecified secondary); grey #7A7A7A (exploratory)",
    "layout": "three fixed columns: forest | exposure definition | values (right-aligned)",
    "outputs": [{"path": str(p), "bytes": p.stat().st_size, "sha256": sha256_file(p)} for p in written],
    "caption": str(cap),
})
print(qc)

# 投稿版凭据：只在非草稿运行时写入，避免被后续草稿重跑覆盖（发布门禁以它为准）
if not IS_DRAFT:
    rec = fig_submission_record(FIG_ID, written, cap,
                                {"data_checked": report["checked"], "data_file": DATA_FILE})
    print("submission record:", rec)
